# Intro

This notebook is used to visualize and manipulate results of the study. It should be run after the `dvc repro` command was successfully executed. 

# Load libs

In [11]:
import os
import pandas as pd
import yaml
from IPython.display import display, Markdown

import autoroot
import autorootcwd

In [12]:
import aml_magic.src.consts as cc
import aml_magic.src.models.metrics as model_metrics
import aml_magic.src.utils.configs as cfg
import aml_magic.src.stages.summarize_results as sumres

# Load data

## Load config

In [13]:
config_yaml = yaml.safe_load(open(f"{cc.MAIN_PARAMS_FILE}.yaml", "r"))
config = cfg.ExperimentConfig(**config_yaml)

## Load summary data

In [14]:
summary_data = pd.read_csv(cc.STUDY_COMPARISON_PATH / "scores_summary.csv")
summary_data.head(3)

,dataset,metric,scope,MAGIC+XGB,GCN,SkipGCN,GCN+XGB,Nenn+XGB,Nenn,SkipGCN+XGB
0,AMLSim 101,F1,Ilicit,0.789 +/- 0.121,0.810 +/- 0.035,0.807 +/- 0.033,0.814 +/- 0.032,0.837 +/- 0.027,0.725 +/- 0.041,0.812 +/- 0.030
1,AMLSim 101,F1,Macro,0.885 +/- 0.065,0.894 +/- 0.020,0.892 +/- 0.019,0.898 +/- 0.018,0.910 +/- 0.015,0.844 +/- 0.024,0.896 +/- 0.017
2,AMLSim 101,Precision,Ilicit,0.914 +/- 0.021,0.724 +/- 0.058,0.719 +/- 0.056,0.786 +/- 0.052,0.828 +/- 0.035,0.592 +/- 0.051,0.779 +/- 0.050


## Load raw scores per dataset

In [15]:
all_raw_scores = []
for dataset in os.listdir(cc.STUDY_RESULTS_PATH):
    dataset_raw_scores = pd.read_csv(cc.STUDY_RESULTS_PATH / dataset / f"{dataset}_MAGIC+xgboost_raw.csv", index_col=0).drop(columns='model')
    dataset_raw_scores["dataset"] = dataset
    all_raw_scores.append(dataset_raw_scores)

all_raw_scores_df = pd.concat(all_raw_scores, ignore_index=True)

# Show results

## Summaries

In [16]:
sorting_order = [f"AMLSim {num}" for num in [31, 51, 101, 201]]
summarized_datasets = summary_data.pivot_table(
    index=['dataset', 'metric', 'scope'],
    aggfunc='max'
)

for ds in sorting_order:
    display(Markdown(f"### {ds}"))
    display(summarized_datasets.loc[ds].style.highlight_max(color='lightgreen', axis=1) )

### AMLSim 31

### AMLSim 51

### AMLSim 101

### AMLSim 201

## Summarize raw scores

Summaries of raw scores below are done without the use of the Bootstrap method.

In [17]:
scores_sorting = [f"amlsim_{num}_CI_SUMMARY" for num in [31, 51, 101, 201]]

In [18]:
all_raw_scores_df.groupby("dataset").agg(['mean', 'std']).loc[scores_sorting]

Macro Precision           Macro Recall            \
                                 mean       std         mean       std   
dataset                                                                  
amlsim_31_CI_SUMMARY         0.921634  0.013582     0.943562  0.021503   
amlsim_51_CI_SUMMARY         0.896833  0.003525     0.904220  0.006476   
amlsim_101_CI_SUMMARY        0.913939  0.004981     0.860400  0.022123   
amlsim_201_CI_SUMMARY        0.927149  0.006939     0.889127  0.005149   

                       Macro F1           Ilicit Precision            \
                           mean       std             mean       std   
dataset                                                                
amlsim_31_CI_SUMMARY   0.930942  0.016431         0.921634  0.013582   
amlsim_51_CI_SUMMARY   0.900454  0.004941         0.896833  0.003525   
amlsim_101_CI_SUMMARY  0.884628  0.015164         0.913939  0.004981   
amlsim_201_CI_SUMMARY  0.907144  0.005710         0.927149  0.006939   

                      Ilicit Recall           Ilicit F1            
                               mean       std      mean       std  
dataset                                                            
amlsim_31_CI_SUMMARY       0.955882  0.043570  0.907407  0.022879  
amlsim_51_CI_SUMMARY       0.849673  0.012336  0.839361  0.008155  
amlsim_101_CI_SUMMARY      0.733660  0.044479  0.789326  0.028205  
amlsim_201_CI_SUMMARY      0.784314  0.009804  0.822633  0.010894

## Calculate boostrap CIs

In [19]:
dataset_bootstrap_scores = {}
metrics = [col for col in all_raw_scores_df.columns if col != 'dataset']
for dataset in all_raw_scores_df.dataset.unique():
    dataset_scores = all_raw_scores_df[all_raw_scores_df.dataset == dataset]
    for metric in metrics:
        boostrap_avg_ci, bootstrap_std_ci = model_metrics.get_confidence_intervals(dataset_scores[metric], n_repeats=config.n_repeats)
        dataset_bootstrap_scores[(dataset, metric)] = {
            "avg": boostrap_avg_ci,
            "std": bootstrap_std_ci
        }
dataset_bootstrap_scores_df = pd.DataFrame(dataset_bootstrap_scores).T

In [20]:
dataset_bootstrap_scores_df.round(3).loc[scores_sorting]

avg    std
amlsim_31_CI_SUMMARY  Macro Precision   0.922  0.058
                      Macro Recall      0.944  0.093
                      Macro F1          0.931  0.071
                      Ilicit Precision  0.922  0.058
                      Ilicit Recall     0.956  0.187
                      Ilicit F1         0.907  0.098
amlsim_51_CI_SUMMARY  Macro Precision   0.897  0.015
                      Macro Recall      0.904  0.028
                      Macro F1          0.900  0.021
                      Ilicit Precision  0.897  0.015
                      Ilicit Recall     0.850  0.053
                      Ilicit F1         0.839  0.035
amlsim_101_CI_SUMMARY Macro Precision   0.914  0.021
                      Macro Recall      0.860  0.095
                      Macro F1          0.885  0.065
                      Ilicit Precision  0.914  0.021
                      Ilicit Recall     0.734  0.191
                      Ilicit F1         0.789  0.121
amlsim_201_CI_SUMMARY Macro Precision   0.927  0.030
                      Macro Recall      0.889  0.022
                      Macro F1          0.907  0.025
                      Ilicit Precision  0.927  0.030
                      Ilicit Recall     0.784  0.042
                      Ilicit F1         0.823  0.047